In [23]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/vecneta/bad-videos/spongebob.mp4
/kaggle/input/datasets/vecneta/bad-videos/bad_talking.mp4
/kaggle/input/competitions/jigsaw-toxic-comment-classification-challenge/train.csv.zip
/kaggle/input/competitions/jigsaw-toxic-comment-classification-challenge/sample_submission.csv.zip
/kaggle/input/competitions/jigsaw-toxic-comment-classification-challenge/test_labels.csv.zip
/kaggle/input/competitions/jigsaw-toxic-comment-classification-challenge/test.csv.zip


In [24]:
import pandas as pd

df = pd.read_csv("/kaggle/input/competitions/jigsaw-toxic-comment-classification-challenge/train.csv.zip")
df_test = pd.read_csv('/kaggle/input/competitions/jigsaw-toxic-comment-classification-challenge/test.csv.zip')
df_test_labels = pd.read_csv('/kaggle/input/competitions/jigsaw-toxic-comment-classification-challenge/test_labels.csv.zip')

categories = ['toxic', 'severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate']


print("عدد الحالات المخالفة لكل فئة:")
print(df[categories].sum())

عدد الحالات المخالفة لكل فئة:
toxic            15294
severe_toxic      1595
obscene           8449
threat             478
insult            7877
identity_hate     1405
dtype: int64


In [25]:
import pandas as pd

clean_samples = df[df[categories].sum(axis=1) == 0][
    ["comment_text"]
].head(5)
clean_samples["label"] = "Clean / Safe"

flagged_samples = []
for cat in categories:
    samples = df[df[cat] == 1][["comment_text"]].head(3)
    samples["label"] = cat
    flagged_samples.append(samples)

mixed_df = pd.concat([clean_samples] + flagged_samples, ignore_index=True)

for i, row in mixed_df.iterrows():
    print(f"[{i+1}] [{row['label']}]")
    print(f"{row['comment_text'][:150]}...\n" + "-" * 50)

[1] [Clean / Safe]
Explanation
Why the edits made under my username Hardcore Metallica Fan were reverted? They weren't vandalisms, just closure on some GAs after I voted...
--------------------------------------------------
[2] [Clean / Safe]
D'aww! He matches this background colour I'm seemingly stuck with. Thanks.  (talk) 21:51, January 11, 2016 (UTC)...
--------------------------------------------------
[3] [Clean / Safe]
Hey man, I'm really not trying to edit war. It's just that this guy is constantly removing relevant information and talking to me through edits instea...
--------------------------------------------------
[4] [Clean / Safe]
"
More
I can't make any real suggestions on improvement - I wondered if the section statistics should be later on, or a subsection of ""types of accid...
--------------------------------------------------
[5] [Clean / Safe]
You, sir, are my hero. Any chance you remember what page that's on?...
--------------------------------------------------
[

In [26]:
df.head()

,id,comment_text,toxic,severe_toxic,obscene,threat,insult,identity_hate
0,0000997932d777bf,Explanation\nWhy the edits made under my usern...,0,0,0,0,0,0
1,000103f0d9cfb60f,D'aww! He matches this background colour I'm s...,0,0,0,0,0,0
2,000113f07ec002fd,"Hey man, I'm really not trying to edit war. It...",0,0,0,0,0,0
3,0001b41b1c6bb37e,"""\nMore\nI can't make any real suggestions on ...",0,0,0,0,0,0
4,0001d958c54c6e35,"You, sir, are my hero. Any chance you remember...",0,0,0,0,0,0


In [27]:
!pip install -q openai-whisper

In [28]:
import re
import string
import nltk
import pandas as pd
import tensorflow as tf
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from tensorflow.keras import layers, models
from tensorflow.keras.layers import TextVectorization

nltk.download('stopwords')
nltk.download('wordnet')

[nltk_data] Downloading package stopwords to /usr/share/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /usr/share/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

In [29]:
lemmatizer = WordNetLemmatizer()

default_stopwords = set(stopwords.words('english'))
important_words = {
    'not', 'no', 'nor', 'against', 'you', 'your', 'yourself', 
    'yours', 'me', 'myself', 'he', 'him', 'she', 'her', 'they', 'them'
}
custom_stopwords = default_stopwords - important_words

contractions_dict = {
    "don't": "do not", "can't": "cannot", "won't": "will not",
    "isn't": "is not", "aren't": "are not", "you're": "you are",
    "i'm": "i am", "it's": "it is", "couldn't": "could not"
}

def advanced_clean_text(text):
    text = str(text).lower()
    
    # Expand contractions
    for contraction, expanded in contractions_dict.items():
        text = text.replace(contraction, expanded)
        
    text = re.sub(r'https?://\S+|www\.\S+', '', text)
    text = re.sub(r'<.*?>+', '', text)
    
    # Clean symbol obfuscation
    text = re.sub(r'@', 'a', text)
    text = re.sub(r'!', 'i', text)
    text = re.sub(r'\$', 's', text)
    text = re.sub(r'\*', '', text)
    
    text = text.translate(str.maketrans('', '', string.punctuation))
    text = re.sub(r'\d+', '', text)
    
    words = text.split()
    cleaned_words = [
        lemmatizer.lemmatize(word) 
        for word in words 
        if word not in custom_stopwords
    ]
    
    return " ".join(cleaned_words)

df=df.drop(columns=['id'])
df['comment_text'] = df['comment_text'].apply(advanced_clean_text)
df.head()

,comment_text,toxic,severe_toxic,obscene,threat,insult,identity_hate
0,explanation edits made username hardcore metal...,0,0,0,0,0,0
1,dawwi he match background colour seemingly stu...,0,0,0,0,0,0
2,hey man really not trying edit war guy constan...,0,0,0,0,0,0
3,cannot make real suggestion improvement wonder...,0,0,0,0,0,0
4,you sir hero chance you remember page thats,0,0,0,0,0,0


In [30]:
import pandas as pd
from sklearn.model_selection import train_test_split

categories = ['toxic', 'severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate']

X_train, X_val, y_train, y_val = train_test_split(
    df['comment_text'].values, 
    df[categories].values, 
    test_size=0.2, 
    random_state=42
)

df_test['comment_text'] = df_test['comment_text'].apply(advanced_clean_text)

X_test = df_test['comment_text'].values
y_test = df_test_labels[categories].values

print("--- Training Set ---")
print(f"X_train Shape: {X_train.shape}")
print(f"y_train Shape: {y_train.shape}\n")

print("--- Validation Set ---")
print(f"X_val Shape:   {X_val.shape}")
print(f"y_val Shape:   {y_val.shape}\n")

print("--- Test Set ---")
print(f"X_test Shape:  {X_test.shape}")
print(f"y_test Shape:  {y_test.shape}")

--- Training Set ---
X_train Shape: (127656,)
y_train Shape: (127656, 6)

--- Validation Set ---
X_val Shape:   (31915,)
y_val Shape:   (31915, 6)

--- Test Set ---
X_test Shape:  (153164,)
y_test Shape:  (153164, 6)


In [31]:
import tensorflow as tf
from tensorflow.keras.layers import TextVectorization

MAX_WORDS = 20000
MAX_LEN = 150

vectorizer = TextVectorization(
    max_tokens=MAX_WORDS,
    output_sequence_length=MAX_LEN, 
    output_mode='int'
)

vectorizer.adapt(X_train)

print(f"Vocabulary built! Total unique words adapted: {len(vectorizer.get_vocabulary())}")

2026-09-02 10:51:04.641856: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


Vocabulary built! Total unique words adapted: 20000


In [32]:
import tensorflow as tf
from sklearn.utils.class_weight import compute_class_weight

weights_per_category = []
for i in range(len(categories)):
    cls_weights = compute_class_weight(
        class_weight='balanced',
        classes=np.unique(y_train[:, i]),
        y=y_train[:, i]
    )
    pos_weight = np.sqrt(cls_weights[1] / cls_weights[0])
    weights_per_category.append(pos_weight)

pos_weights_tensor = tf.constant(weights_per_category, dtype=tf.float32)

def weighted_binary_crossentropy(y_true, y_pred):
    y_true = tf.cast(y_true, tf.float32)
    epsilon = tf.keras.backend.epsilon()
    y_pred = tf.clip_by_value(y_pred, epsilon, 1.0 - epsilon)
    bce = - (y_true * tf.math.log(y_pred) * pos_weights_tensor + (1.0 - y_true) * tf.math.log(1.0 - y_pred))
    return tf.reduce_mean(bce)


In [33]:
model = models.Sequential([
    layers.Input(shape=(1,), dtype=tf.string),
    vectorizer,
    layers.Embedding(input_dim=MAX_WORDS, output_dim=128),
    layers.Bidirectional(layers.LSTM(64, return_sequences=True)),
    layers.GlobalAveragePooling1D(),
    layers.Dense(64, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(6, activation='sigmoid')
])

In [34]:
model.compile(
    loss=weighted_binary_crossentropy,
    optimizer='adam',
    metrics=['accuracy', tf.keras.metrics.AUC(name='auc')]
)

print("Training started...\n")
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=5,
    batch_size=64,
    verbose=1 
)
print("\nTraining finished!\n")
valid_idx = (y_test != -1).all(axis=1)
X_test_clean = X_test[valid_idx]
y_test_clean = y_test[valid_idx]

OPTIMAL_THRESHOLD = 0.70
y_pred_probs = model.predict(X_test_clean, batch_size=64, verbose=0)

print("="*50)
print(f"   DETAILED EVALUATION REPORT (Threshold = {OPTIMAL_THRESHOLD})   ")
print("="*50 + "\n")

for i, cat in enumerate(categories):
    y_true_cat = y_test_clean[:, i]
    y_pred_cat = (y_pred_probs[:, i] >= OPTIMAL_THRESHOLD).astype(int)
    
    tp = np.sum((y_true_cat == 1) & (y_pred_cat == 1))
    fp = np.sum((y_true_cat == 0) & (y_pred_cat == 1))
    fn = np.sum((y_true_cat == 1) & (y_pred_cat == 0))
    
    prec = tp / (tp + fp + 1e-7)
    rec = tp / (tp + fn + 1e-7)
    f1 = 2 * (prec * rec) / (prec + rec + 1e-7)
    
    auc_metric = tf.keras.metrics.AUC()
    auc_metric.update_state(y_true_cat, y_pred_probs[:, i])
    auc = auc_metric.result().numpy()
    
    print(f"📌 Class: {cat.upper()}")
    print(f"   ├── AUC:       {auc:.4f}")
    print(f"   ├── Precision: {prec:.4f}")
    print(f"   ├── Recall:    {rec:.4f}")
    print(f"   └── F1-Score:  {f1:.4f}")
    print("-" * 50)

Training started...

Epoch 1/5
1995/1995 ━━━━━━━━━━━━━━━━━━━━ 370s 183ms/step - accuracy: 0.8807 - auc: 0.9448 - loss: 0.2248 - val_accuracy: 0.9941 - val_auc: 0.9791 - val_loss: 0.1484
Epoch 2/5
1995/1995 ━━━━━━━━━━━━━━━━━━━━ 379s 190ms/step - accuracy: 0.9795 - auc: 0.9822 - loss: 0.1365 - val_accuracy: 0.9941 - val_auc: 0.9832 - val_loss: 0.1396
Epoch 3/5
1995/1995 ━━━━━━━━━━━━━━━━━━━━ 364s 182ms/step - accuracy: 0.9628 - auc: 0.9871 - loss: 0.1184 - val_accuracy: 0.9941 - val_auc: 0.9798 - val_loss: 0.1382
Epoch 4/5
1995/1995 ━━━━━━━━━━━━━━━━━━━━ 364s 182ms/step - accuracy: 0.9567 - auc: 0.9904 - loss: 0.1010 - val_accuracy: 0.9941 - val_auc: 0.9773 - val_loss: 0.1438
Epoch 5/5
1995/1995 ━━━━━━━━━━━━━━━━━━━━ 362s 182ms/step - accuracy: 0.8766 - auc: 0.9926 - loss: 0.0867 - val_accuracy: 0.9941 - val_auc: 0.9714 - val_loss: 0.1706

Training finished!

   DETAILED EVALUATION REPORT (Threshold = 0.7)   

📌 Class: TOXIC
   ├── AUC:       0.9488
   ├── Precision: 0.4685
   ├── Recall:  

In [41]:
import numpy as np
import tensorflow as tf
import whisper
speech_model = whisper.load_model("base")

video_path = "/kaggle/input/datasets/vecneta/bad-videos/bad_talking.mp4"
result = speech_model.transcribe(video_path)

extracted_text = result["text"]
print("extracted audio:")
print(extracted_text)

input_text = np.array([extracted_text], dtype=object)

probabilities = model.predict(input_text, verbose=0)[0]

top_class_index = np.argmax(probabilities)
top_category = categories[top_class_index].upper()
top_score = probabilities[top_class_index] * 100

print("=" * 50)
print(f" Extracted Text: \"{extracted_text.strip()}\"")
print("=" * 50)
print(" Class Probabilities:\n")

for cat, prob in zip(categories, probabilities):
    print(f"• {cat.upper():<15}: {prob * 100:6.2f}%")

print("-" * 50)
print(f" Predicted Category (Highest Probability): {top_category} ({top_score:.2f}%)")
print("=" * 50)

/usr/local/lib/python3.12/dist-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")


extracted audio:
 I don't let people screw when you have black and gold eyes. When you like people screw, when you touch the world, and fucking reach the right, but you're husband can't deal with all I can do to understand. When you just sit here and say, I like nothing. Yes, it's us. It has everything to do with it. If everything is to you to sit here and press, look at me when I talk to you.
 Extracted Text: "I don't let people screw when you have black and gold eyes. When you like people screw, when you touch the world, and fucking reach the right, but you're husband can't deal with all I can do to understand. When you just sit here and say, I like nothing. Yes, it's us. It has everything to do with it. If everything is to you to sit here and press, look at me when I talk to you."
 Class Probabilities:

• TOXIC          :  98.71%
• SEVERE_TOXIC   :  72.18%
• OBSCENE        :  89.35%
• THREAT         :  80.14%
• INSULT         :  89.79%
• IDENTITY_HATE  :  84.78%
--------------------

In [44]:
import numpy as np
import tensorflow as tf

video_path2 = "/kaggle/input/datasets/vecneta/bad-videos/spongebob.mp4"
result2 = speech_model.transcribe(video_path2)

extracted_text2 = result2["text"]
print("extracted audio:")
print(extracted_text2)

input_text2 = np.array([extracted_text2], dtype=object)

probabilities2 = model.predict(input_text2, verbose=0)[0]

top_class_index2 = np.argmax(probabilities2)
top_category2 = categories[top_class_index2].upper()
top_score2 = probabilities2[top_class_index2] * 100

print("=" * 50)
print(f" Extracted Text: \"{extracted_text2.strip()}\"")
print("=" * 50)
print(" Class Probabilities:\n")
for cat, prob in zip(categories, probabilities2):
    print(f"• {cat.upper():<15}: {prob * 100:6.2f}%")

print("-" * 50)
print(f" Predicted Category (Highest Probability): {top_category2} ({top_score2:.2f}%)")
print("=" * 50)

/usr/local/lib/python3.12/dist-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")


extracted audio:
 You got me in trouble. You got me moved to the back of the room. Get cost me one of my good noodle stars. Who cares about a stupid star? Shut the fuck up, Patrick! This is why you're a fucking stupid star. Well fuck you...
 Extracted Text: "You got me in trouble. You got me moved to the back of the room. Get cost me one of my good noodle stars. Who cares about a stupid star? Shut the fuck up, Patrick! This is why you're a fucking stupid star. Well fuck you..."
 Class Probabilities:

• TOXIC          :  99.58%
• SEVERE_TOXIC   :  56.20%
• OBSCENE        :  98.74%
• THREAT         :   7.17%
• INSULT         :  94.84%
• IDENTITY_HATE  :  34.17%
--------------------------------------------------
 Predicted Category (Highest Probability): TOXIC (99.58%)
